In [ ]:
!pip uninstall -y \
torch \
torchvision \
torchaudio \
transformers \
accelerate \
flash-attn

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1


In [ ]:
!pip install -q \
torch==2.10.0 \
torchvision==0.25.0 \
transformers==4.57.1 \
safetensors \
einops==0.8.2 \
addict==2.4.0 \
easydict==1.13 \
pymupdf==1.27.2.2 \
matplotlib==3.10.8 \
psutil \
pillow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires accelerate>=0.21.0, which is not installed.


In [ ]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
TorchVision: 0.25.0+cu128
CUDA: 12.8
GPU: Tesla T4


In [ ]:
import transformers

print(transformers.__version__)

4.57.1


In [ ]:
from transformers import AutoTokenizer

MODEL = "baidu/Unlimited-OCR"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)

print("Tokenizer OK")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer OK


In [ ]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained(
    MODEL,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)

model.eval().cuda()

print("Model OK")

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model OK


In [ ]:
import tempfile
import os
import fitz

In [ ]:
def pdf_to_images(pdf_path, dpi=300):

    doc = fitz.open(pdf_path)

    tmp_dir = tempfile.mkdtemp()

    image_paths = []

    mat = fitz.Matrix(dpi/72, dpi/72)

    for i, page in enumerate(doc):

        img_path = os.path.join(
            tmp_dir,
            f"page_{i+1:04d}.png"
        )

        page.get_pixmap(matrix=mat).save(img_path)

        image_paths.append(img_path)

    doc.close()

    return image_paths

In [ ]:
batch_size = 20

for start in range(0, len(images), batch_size):
    batch = images[start:start + batch_size]

    model.infer_multi(
        tokenizer,
        prompt="<image>Multi page parsing.",
        image_files=batch,
        output_path=f"output/batch_{start//batch_size}",
        image_size=1024,
        max_length=32768,
        no_repeat_ngram_size=35,
        ngram_window=1024,
        save_results=True
    )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>image [0, 0, 263, 453]<|/det|>
<|det|>title [302, 226, 934, 352]<|/det|>WHO recommendations for care of the preterm or low-birth-weight infant
<|det|>footer [483, 875, 571, 929]<|/det|>[Non-Text]
<|det|>footer [576, 882, 739, 904]<|/det|>World Health
<|det|>footer [576, 904, 739, 927]<|/det|>Organization
<PAGE><|det|>title [207, 224, 837, 352]<|/det|>WHO recommendations for care of the preterm or low-birth-weight infant
<|det|>footer [378, 874, 465, 929]<|/det|>[Non-Text]
<|det|>footer [475, 882, 537, 925]<|/det|>[Non-Text]
<|det|>footer [538, 882, 655, 903]<|/det|>World Health
<|det|>footer [538, 904, 655, 925]<|/det|>Organization
<PAGE><|det|>title [208, 225, 837, 352]<|/det|>WHO recommendations for care of the preterm or low-birth-weight infant
<|det|>footer [378, 874, 465, 928]<|/det|>[Non-Text]
<|det|>footer [475, 882, 634, 903]<|/det|>World Health
<|det|>footer [475, 904, 634, 925]<|/det|>Organization
<PAGE><|det|>text [115, 69, 632, 84]<|/det|>WHO recommendations fo

other_page_0: 100%|██████████| 4/4 [00:00<00:00, 40041.09it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_1: 100%|██████████| 5/5 [00:00<00:00, 27271.16it/s]
image_page_2: 0it [00:00, ?it/s]
other_page_2: 100%|██████████| 4/4 [00:00<00:00, 38391.80it/s]
image_page_3: 0it [00:00, ?it/s]
other_page_3: 100%|██████████| 14/14 [00:00<00:00, 75670.43it/s]
image_page_4: 0it [00:00, ?it/s]
other_page_4: 100%|██████████| 48/48 [00:00<00:00, 95279.98it/s]
image_page_5: 0it [00:00, ?it/s]
other_page_5: 100%|██████████| 11/11 [00:00<00:00, 97748.61it/s]
image_page_6: 0it [00:00, ?it/s]
other_page_6: 100%|██████████| 8/8 [00:00<00:00, 48489.06it/s]
image_page_7: 0it [00:00, ?it/s]
other_page_7: 100%|██████████| 4/4 [00:00<00:00, 38304.15it/s]
image_page_8: 0it [00:00, ?it/s]
other_page_9: 100%|██████████| 13/13 [00:00<00:00, 51488.15it/s]
image_page_10: 0it [00:00, ?it/s]
other_page_10: 100%|██████████| 12/12 [00:00<00:00, 41053.55it/s]
image_page_11: 0it [00:00, ?it/s]
other_page_11: 100%|███████

<PAGE><|det|>aside_text [36, 545, 58, 918]<|/det|>WHOop.commations the peer review article content
<|det|>text [113, 68, 486, 452]<|/det|>For effects (benefits and harms), evidence was derived from systematic reviews of randomized controlled trials (RCTs) where possible. If reviews of RCTs were not available, then systematic reviews of non-randomized studies of interventions were used. An overview of systematic reviews was compiled to identify all eligible systematic reviews that had been conducted in the last three years (26). If systematic reviews were not available, they were commissioned from expert systematic review groups. All commissioned systematic reviews followed standard methods, including: a standard protocol published in advance; a clear PICO question; criteria for identification of studies, including search strategies for different bibliographic databases; methods for assessing risk of bias; and a data analysis plan. The protocols were reviewed and approved by members of 

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 12/12 [00:00<00:00, 59705.40it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_1: 100%|██████████| 22/22 [00:00<00:00, 74175.79it/s]
image_page_2: 0it [00:00, ?it/s]
other_page_2: 100%|██████████| 13/13 [00:00<00:00, 52529.82it/s]
image_page_3: 0it [00:00, ?it/s]
other_page_3: 100%|██████████| 9/9 [00:00<00:00, 87179.53it/s]
image_page_4: 0it [00:00, ?it/s]
other_page_4: 100%|██████████| 25/25 [00:00<00:00, 88637.02it/s]
image_page_5: 0it [00:00, ?it/s]
other_page_5: 100%|██████████| 16/16 [00:00<00:00, 63550.06it/s]
image_page_6: 0it [00:00, ?it/s]
other_page_6: 100%|██████████| 22/22 [00:00<00:00, 69119.62it/s]
image_page_7: 0it [00:00, ?it/s]
other_page_7: 100%|██████████| 17/17 [00:00<00:00, 66888.53it/s]
image_page_8: 0it [00:00, ?it/s]
other_page_8: 100%|██████████| 13/13 [00:00<00:00, 64834.66it/s]
image_page_9: 0it [00:00, ?it/s]
other_page_9: 100%|██████████| 22/22 [00:00<00:00, 27835.50it/s]
image_page_10: 0it [00

...

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 21/21 [00:00<00:00, 73278.19it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_1: 100%|██████████| 6/6 [00:00<00:00, 18558.87it/s]
image_page_2: 0it [00:00, ?it/s]
other_page_2: 100%|██████████| 20/20 [00:00<00:00, 76121.67it/s]
image_page_3: 0it [00:00, ?it/s]
other_page_3: 100%|██████████| 20/20 [00:00<00:00, 62648.30it/s]
image_page_4: 0it [00:00, ?it/s]
other_page_4: 100%|██████████| 14/14 [00:00<00:00, 45169.43it/s]
image_page_5: 0it [00:00, ?it/s]
other_page_5: 100%|██████████| 32/32 [00:00<00:00, 42744.50it/s]
image_page_6: 0it [00:00, ?it/s]
other_page_6: 100%|██████████| 20/20 [00:00<00:00, 73584.28it/s]
image_page_7: 0it [00:00, ?it/s]
other_page_7: 100%|██████████| 29/29 [00:00<00:00, 119601.59it/s]
image_page_8: 0it [00:00, ?it/s]
other_page_8: 100%|██████████| 33/33 [00:00<00:00, 85386.82it/s]
image_page_9: 0it [00:00, ?it/s]
other_page_9: 100%|██████████| 22/22 [00:00<00:00, 39199.10it/s]
image_page_10: 0it [0

<PAGE><|det|>title [115, 69, 237, 82]<|/det|>Other outcomes
<|det|>text [113, 85, 491, 325]<|/det|>There was little or no effect on infant temperament at 6 months of age (SMD 0.26, 95% CI -0.29 to 0.81; 2 trials, 155 participants). There was an increase in mother–infant interaction at 6 weeks (MD 1.8, 95% CI 0.21 to 3.81; 1 trial, 142 participants), 3 months (MD 0.8, 95% CI 0.6 to 2.2; 1 trial, 196 participants) and 6 months of age (MD 0.21, 95% CI 0.11 to 0.67; 1 trial, 63 participants), but there was little to no effect at follow-up at 12 months of age (MD 0.1, 95% CI -0.01 to 0.21; 1 trial, 93 participants). There was little to no effect on duration of exclusive breastfeeding (EBF) (MD 2.0, 95% CI -5.48 to 9.48; 1 trial, 128 participants), but there was an increase in EBF at 2–3 months (RR 1.71, 95% CI 1.26 to 2.3; 2 trials, 244 participants).
<|det|>title [114, 340, 449, 370]<|/det|>Effectiveness: Comparison 2 – Peer support versus usual care
<|det|>title [114, 370, 430, 383]<|/det

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 25/25 [00:00<00:00, 61141.46it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_1: 100%|██████████| 19/19 [00:00<00:00, 16086.35it/s]
image_page_2: 0it [00:00, ?it/s]
other_page_2: 100%|██████████| 3/3 [00:00<00:00, 28339.89it/s]
image_page_3: 0it [00:00, ?it/s]
other_page_3: 100%|██████████| 20/20 [00:00<00:00, 87563.76it/s]
image_page_4: 0it [00:00, ?it/s]
other_page_4: 100%|██████████| 25/25 [00:00<00:00, 74419.87it/s]
image_page_5: 0it [00:00, ?it/s]
other_page_5: 100%|██████████| 9/9 [00:00<00:00, 75648.77it/s]
image_page_6: 0it [00:00, ?it/s]
other_page_6: 100%|██████████| 18/18 [00:00<00:00, 24393.37it/s]
image_page_7: 0it [00:00, ?it/s]
other_page_7: 100%|██████████| 23/23 [00:00<00:00, 19289.94it/s]
image_page_8: 0it [00:00, ?it/s]
other_page_8: 100%|██████████| 27/27 [00:00<00:00, 70121.49it/s]
image_page_9: 0it [00:00, ?it/s]
other_page_9: 100%|██████████| 24/24 [00:00<00:00, 68853.14it/s]
image_page_10: 0it [00:0

<PAGE><|det|>list [108, 79, 500, 905]<|/det|>
<|det|>ref_text [117, 84, 474, 180]<|/det|>51. Vesel L, ten Asbroek AH, Manu A, Soremekun S, Tawiah Agyemang C, Okyere E, et al. Promoting skin-to-skin care for low birthweight babies: findings from the Ghana Newhints cluster-randomised trial. Trop Med Int Health. 2013;18(8):952-61. doi:10.1111/tmi.12134.
<|det|>ref_text [117, 197, 484, 293]<|/det|>52. Requejo J, Diaz T, Park L, Chou D, Choudhury A, Guthold R, et al. Assessing coverage of interventions for reproductive, maternal, newborn, child, and adolescent health and nutrition. BMJ. 2020;368:16915. doi:10.1136/bmj.16915.
<|det|>ref_text [117, 309, 484, 420]<|/det|>53. Maternal and newborn - Coverage. In: Maternal, newborn, child and adolescent health and ageing: data portal [website]. Geneva: World Health Organization; 2022 (www.who.int/data/maternal-newborn-child-adolescent-ageing/maternal-and-newborn-data/maternal-and-newborn---coverage, accessed 11 April 2022).
<|det|>ref_text [117, 

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/VLM

In [ ]:
!cp -r /content/output /content/drive/MyDrive/VLM

In [ ]:
import os

for f in os.listdir("/content/drive/MyDrive/VLM/output"):
    print(f)

batch_0
batch_1
batch_2
batch_3
batch_4
batch_5
batch_6


In [ ]:
from pathlib import Path
import re
import os
from collections import Counter

INPUT_DIR = Path("/content/drive/MyDrive/VLM/output")
OUTPUT_DIR = Path("/content/drive/MyDrive/VLM/postprocess")

OUTPUT_DIR.mkdir(exist_ok=True)

print(INPUT_DIR)

/content/drive/MyDrive/VLM/output


In [ ]:
batch_files = sorted(INPUT_DIR.glob("batch_*/result.md"))

print(len(batch_files))

for f in batch_files:
    print(f)

6
/content/drive/MyDrive/VLM/output/batch_0/result.md
/content/drive/MyDrive/VLM/output/batch_1/result.md
/content/drive/MyDrive/VLM/output/batch_2/result.md
/content/drive/MyDrive/VLM/output/batch_3/result.md
/content/drive/MyDrive/VLM/output/batch_4/result.md
/content/drive/MyDrive/VLM/output/batch_5/result.md


In [ ]:
def clean_markdown(text):

    # remove page token
    text = re.sub(r"<PAGE>", "", text)

    # remove OCR image
    text = re.sub(r"!\[\]\(.*?\)", "", text)

    # remove Non-Text
    text = re.sub(r"\[Non-Text\]", "", text)

    # remove trailing spaces
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

    # collapse many blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
counter = Counter()

for file in batch_files:

    txt = clean_markdown(file.read_text())

    for line in txt.splitlines():

        line = line.strip()

        if len(line) > 5:
            counter[line] += 1

In [ ]:
for line, freq in counter.most_common(30):

    if freq > 5:
        print(freq, line)

39 Chapter 3. Evidence and recommendations
33 Sources and characteristics of the evidence
32 Critical outcomes
32 Subgroup analyses
28 Background and definitions
28 Summary of the evidence
27 Recommendation and remarks
26 Remarks
26 Summary of judgements
25 Values and acceptability
24 Resources required and implementation considerations
24 Feasibility and equity
23 Other outcomes
22 Organization of care
22 Infrastructure, equipment and supplies
22 Workforce, training, supervision and monitoring
18 Evidence-to-Decision summary
16 The effect of gestational age and birth weight could not be assessed as there were insufficient trials for any critical outcome.
12 - The recommendation is conditional on shared decision-making with parents; this includes informing parents about the benefits and risks and the need for further research.
12 Outcomes - All-cause mortality, morbidity, growth, neurodevelopment at latest follow-up
10 Justification
9 OVERVIEW
9 Health workers at all levels can support

In [ ]:
COMMON_LINES = {
    line
    for line, freq in counter.items()
    if freq > 5
}

In [ ]:
def remove_common_lines(text):

    lines = []

    for line in text.splitlines():

        if line.strip() not in COMMON_LINES:
            lines.append(line)

    return "\n".join(lines)

In [ ]:
def merge_lines(text):

    lines = text.splitlines()

    merged = []

    for line in lines:

        line = line.strip()

        if not merged:
            merged.append(line)
            continue

        prev = merged[-1]

        if (
            prev.endswith((".", ":", ";", "?", "!"))
            or line.startswith("#")
            or re.match(r"^[A-Z]\.", line)
        ):
            merged.append(line)

        else:
            merged[-1] += " " + line

    return "\n".join(merged)

In [ ]:
def recover_hyphen(text):
    # Replaces hyphens at the end of a line followed by text on the next line
    # This is a common OCR error where words are split by hyphens.
    return re.sub(r"(-\n)(?!\n)", "", text)

clean_batches = []

for file in batch_files:

    text = file.read_text()

    text = clean_markdown(text)

    text = remove_common_lines(text)

    text = recover_hyphen(text)

    text = merge_lines(text)

    out = OUTPUT_DIR / (file.parent.name + ".md")

    out.write_text(text)

    clean_batches.append(out)

print("done")

done


In [ ]:
merged = ""

for f in clean_batches:

    merged += f.read_text()

    merged += "\n\n"

merged_path = OUTPUT_DIR / "merged.md"

merged_path.write_text(merged)

print(merged_path)

/content/drive/MyDrive/VLM/postprocess/merged.md


In [ ]:
MAX_CHARS = 12000

text = merged_path.read_text()

chunks = []

for i in range(0, len(text), MAX_CHARS):

    chunks.append(text[i:i+MAX_CHARS])

print(len(chunks))

30


In [ ]:
PROMPT = """
You are an OCR post-processing assistant.

Do NOT summarize.

Tasks:

- Correct OCR mistakes.
- Merge broken sentences.
- Preserve every sentence.
- Preserve markdown.
- Preserve headings.
- Preserve lists.
- Preserve tables.
- Do not hallucinate.
- Do not remove information.

OCR TEXT

{}
"""

In [ ]:
!pip uninstall -y transformers
!pip install -U transformers accelerate sentencepiece safetensors

Found existing installation: transformers 5.14.1
Uninstalling transformers-5.14.1:
  Successfully uninstalled transformers-5.14.1
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)


In [ ]:
import transformers
print(transformers.__version__)

5.14.1


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:
def generate(
    prompt,
    max_new_tokens=2048,
):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # tokenizer.apply_chat_template returns a dictionary (BatchEncoding)
    # containing 'input_ids' and 'attention_mask' as tensors.
    inputs_dict = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Move each tensor in the dictionary to the model's device
    input_ids = inputs_dict['input_ids'].to(model.device)
    attention_mask = inputs_dict['attention_mask'].to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
responses = []

for i, chunk in enumerate(chunks):

    print(f"Processing {i+1}/{len(chunks)}")

    response = generate(
        PROMPT.format(chunk),
        max_new_tokens=1024
    )

    responses.append(response)

Processing 1/118
Processing 2/118
Processing 3/118
Processing 4/118
Processing 5/118
Processing 6/118
Processing 7/118
Processing 8/118
Processing 9/118
Processing 10/118
Processing 11/118
Processing 12/118
Processing 13/118
Processing 14/118
Processing 15/118
Processing 16/118
Processing 17/118
Processing 18/118
Processing 19/118
Processing 20/118
Processing 21/118
Processing 22/118
Processing 23/118
Processing 24/118
Processing 25/118
Processing 26/118
Processing 27/118
Processing 28/118
Processing 29/118
Processing 30/118
Processing 31/118
Processing 32/118
Processing 33/118
Processing 34/118
Processing 35/118
Processing 36/118
Processing 37/118
Processing 38/118
Processing 39/118
Processing 40/118
Processing 41/118
Processing 42/118
Processing 43/118
Processing 44/118
Processing 45/118
Processing 46/118
Processing 47/118
Processing 48/118
Processing 49/118
Processing 50/118
Processing 51/118
Processing 52/118
Processing 53/118
Processing 54/118
Processing 55/118
Processing 56/118
P

In [ ]:
MAX_CHARS = 3000

text = merged_path.read_text()

chunks = []

for i in range(0, len(text), MAX_CHARS):

    chunks.append(text[i:i+MAX_CHARS])

print(len(chunks))

118


In [ ]:
final_text = ""

for response in responses:

    final_text += response

(Path(OUTPUT_DIR) / "final.md").write_text(final_text)

467879

In [ ]:
processed_md = "\n\n".join(responses)

with open("processed_result.md", "w", encoding="utf-8") as f:
    f.write(processed_md)

print("Saved processed_result.md")

Saved processed_result.md


In [ ]:
import re

clean_lines = []

remove_patterns = [
    r"^Assistant Response",
    r"^Corrected OCR",
    r"^Your response",
    r"^The response",
    r"^Explanation",
    r"^Here is",
    r"^Output:",
]

for line in lines:

    line = line.strip()

    if not line:
        clean_lines.append("")
        continue

    skip = False

    for p in remove_patterns:
        if re.match(p, line, re.IGNORECASE):
            skip = True
            break

    if skip:
        continue

    clean_lines.append(line)

print(len(clean_lines))

3077


In [ ]:
import re

document = []

current = {
    "chapter": None,
    "section": None,
    "subsection": None
}

paragraph = []


def save_chunk():
    global paragraph

    if paragraph:
        document.append({
            "text": " ".join(paragraph),
            "metadata": {
                "chapter": current["chapter"],
                "section": current["section"],
                "subsection": current["subsection"],
            }
        })

        paragraph = []


for line in lines:

    line = line.strip()

    if not line:
        continue


    # Chapter
    if re.match(r"^# ", line):
        save_chunk()

        current["chapter"] = line.replace("# ", "").strip()
        current["section"] = None
        current["subsection"] = None


    # Section
    elif re.match(r"^## ", line):
        save_chunk()

        current["section"] = line.replace("## ", "").strip()
        current["subsection"] = None


    # Subsection
    elif re.match(r"^### ", line):
        save_chunk()

        current["subsection"] = line.replace("### ", "").strip()


    else:
        paragraph.append(line)


save_chunk()

In [ ]:
from pprint import pprint

for x in document[:30]:

    pprint(x)

    print("-"*80)

{'metadata': {'chapter': None, 'section': None, 'subsection': None},
 'text': 'WHO recommendations for care of the preterm or low-birth-weight '
         'infant World Health Organization WHO recommendations for care of the '
         'preterm or low-birth-weight infant World Health Organization WHO '
         'recommendations for care of the preterm or low birth weight infant '
         'World Health Organization ISBN 978-92-4-005826-2 (electronic '
         'version) ISBN 978-92-4-005827-9 (print version) © World Health '
         'Organization 2022 Some rights reserved. This work is available under '
         'the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO '
         'licence (CC BY-NC-SA 3.0 IGO; '
         'https://creativecommons.org/licenses/by-nc-sa/3.0/igo). Under the '
         'terms of this licence, you may copy, redistribute and adapt the work '
         'for non-commercial purposes, provided the work is appropriately '
         'cited, as indicated belo

In [ ]:
import pysbd

segmenter = pysbd.Segmenter(
    language="en",
    clean=False
)

sentences = []

for para in document:

    sents = segmenter.segment(para["text"])

    for s in sents:

        s = s.strip()

        if s:

            sentences.append({

                "text": s,

                "metadata": para["metadata"]

            })

print(len(sentences))

2018


In [ ]:
chunks = []

window = 6
overlap = 2

i = 0

while i < len(sentences):

    part = sentences[i:i+window]

    text = " ".join(x["text"] for x in part)

    metadata = part[0]["metadata"]

    chunks.append({

        "text": text,

        "metadata": metadata

    })

    i += window-overlap

print(len(chunks))

505


In [ ]:
DOCUMENT = "WHO recommendations for care of the preterm or low-birth-weight infant"

for chunk in chunks:

    m = chunk["metadata"]

    prefix = f"""Document: {DOCUMENT}

Chapter: {m['chapter']}

Section: {m['section']}

Subsection: {m['subsection']}
"""

    chunk["embedding_text"] = prefix + "\n\n" + chunk["text"]

In [ ]:
print(chunks[20]["embedding_text"])

Document: WHO recommendations for care of the preterm or low-birth-weight infant

Chapter: None

Section: None

Subsection: None


: several species LAZ: length-for-age z score UNICEF: United Nations Children’s Fund LMIC: low- or middle-income country USS: United States dollar LMP: last menstrual period USAID: United States Agency for International Development MCA: Department of Maternal, Newborn, Child and Adolescent Health and Ageing (at WHO) WHO: World Health Organization vi Glossary LBW: Low birth weight VLBW: Very low birth weight Extremely LBW: Extremely low birth weight Term gestation: Birth between 37 0/7 - 41 6/7 weeks of gestation Preterm: Birth before 37 0/7 weeks of gestation Very preterm: Birth before 32 0/7 weeks of gestation Extremely preterm: Birth before 28 0/7 weeks of gestation Post-term: Birth at or after 42 0/7 weeks of gestation Chronological (or postnatal) age: Age since birth Corrected age: Age adjusted for prematurity PMA: Postmenstrual age Stunting: Height-for

In [ ]:
import json

output_path = "processed_document.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for item in document:
        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            + "\n"
        )

print(f"✅ Saved: {output_path}")

✅ Saved: processed_document.jsonl


In [ ]:
import json

data = []

with open("processed_document.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print("Số document:", len(data))

print(data[0])

Số document: 85
{'text': 'WHO recommendations for care of the preterm or low-birth-weight infant World Health Organization WHO recommendations for care of the preterm or low-birth-weight infant World Health Organization WHO recommendations for care of the preterm or low birth weight infant World Health Organization ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © World Health Organization 2022 Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo). Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial purposes, provided the work is appropriately cited, as indicated below. In any use of this work, there should be no suggestion that WHO endorses any specific organization, products or services. The use of the WHO logo is not permitted. If you adapt the work, then yo